# Servidor Único de Inferencia en Google Colab (M1 + M2 + M3 + Pasajeros)
Ejecuta todas las celdas en orden (`Entorno de ejecución -> Ejecutar todas`). Funciona tanto en **GPU T4** como en **CPU** si alcanzaste el límite diario de GPU, y registra automáticamente el túnel en Render.


In [ ]:
!nvidia-smi -L || echo 'Ejecutando en CPU (el sistema funciona igual, solo con mayor latencia por frame)'


In [ ]:
import os
if not os.path.exists('/content/Control_Peaje'):
    !git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje
else:
    !git -C /content/Control_Peaje fetch --all && git -C /content/Control_Peaje reset --hard origin/main
%cd /content/Control_Peaje
!pip -q install -r inferencia/requirements.txt gdown


In [ ]:
from google.colab import drive
import os

drive_ok = False
try:
    drive.mount('/content/drive')
    drive_ok = os.path.exists('/content/drive/MyDrive')
except Exception as e:
    print('⚠️ No se pudo montar Drive en esta cuenta:', e)
    print('Continuando con almacenamiento local /content/peaje_local')

base_path = '/content/drive/MyDrive/peaje' if drive_ok else '/content/peaje_local'
os.makedirs(f'{base_path}/evidencias', exist_ok=True)
os.makedirs(f'{base_path}/modelos', exist_ok=True)

diag_file = f'{base_path}/evidencias/diagnostico_inferencia.txt'
!python scripts/diagnostico.py > {diag_file}
!cat {diag_file}


In [ ]:
import os
from google.colab import userdata

try:
    os.environ['API_KEY'] = (userdata.get('INFERENCE_API_KEY') or '').strip()
except Exception:
    os.environ['API_KEY'] = ''
    print('Aviso: INFERENCE_API_KEY no definido en Secrets; el endpoint aceptara peticiones sin clave.')

# Si usas otra cuenta de Google sin Drive montado y subiste tus .pt con enlace compartido,
# puedes pegar aqui sus IDs de Google Drive (opcional) para descargarlos con gdown:
M2_DRIVE_ID = ''  # ej. '1AbC...'

modelos_dir = f'{base_path}/modelos'
m2_pt = f'{modelos_dir}/m2_seg.pt'
# Por defecto usamos el modelo entrenado m3_llantas.pt integrado en el repositorio
m3_pt = '/content/Control_Peaje/inferencia/modelos/m3/m3_llantas.pt'
pas_pt = f'{modelos_dir}/pasajeros_seg.pt'

if not os.path.exists(m2_pt) and M2_DRIVE_ID:
    !gdown --id {M2_DRIVE_ID} -O {m2_pt}

os.environ['M2_PESOS'] = m2_pt
os.environ['M3_PESOS'] = m3_pt
os.environ['PASAJEROS_PESOS'] = pas_pt
os.environ['RENDER_WEB_URL'] = 'https://peaje-web.onrender.com'
print('Pesos M2 propios:', os.path.exists(m2_pt), '| Pesos M3 (m3_llantas.pt):', os.path.exists(m3_pt))


In [ ]:
import threading, time, requests, uvicorn
from inferencia.servidor import app

PUERTO = 8000
config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
servidor = uvicorn.Server(config)
hilo = threading.Thread(target=servidor.run, daemon=True)
hilo.start()

# Esperar a que descargue pesos y abra completamente el puerto 8000
for i in range(45):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=3)
        if r.status_code == 200:
            print('✅ Servidor local listo:', r.json().get('modelos'), '| fuente M2:', r.json().get('fuente_m2'))
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('⚠️ El servidor tardó más de 90s en arrancar; revisa la salida superior.')


In [ ]:
import os, re, subprocess, time, requests
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
log_cf = open('/tmp/cloudflared.log', 'w')
proc_cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=log_cf, stderr=subprocess.STDOUT)

url_tunel = None
for _ in range(35):
    time.sleep(1)
    txt = open('/tmp/cloudflared.log').read()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url_tunel = m.group(0)
        break

print('====================================================')
print('🔗 URL DEL TÚNEL CLOUDFLARE:', url_tunel)
print('====================================================')

if url_tunel:
    # Esperar a que DNS de Cloudflare propague y responda /health antes de avisar a Render
    tunel_vivo = False
    for intento in range(12):
        try:
            rh = requests.get(f'{url_tunel}/health', timeout=8)
            if rh.status_code == 200:
                tunel_vivo = True
                break
        except Exception:
            pass
        time.sleep(3)

    render_url = os.environ.get('RENDER_WEB_URL', 'https://peaje-web.onrender.com').rstrip('/')
    conectado = False
    if tunel_vivo:
        for _ in range(3):
            try:
                r_reg = requests.post(
                    f'{render_url}/estado/auto-colab',
                    json={'inference_url': url_tunel, 'api_key': os.environ.get('API_KEY', '')},
                    timeout=25,
                )
                if r_reg.status_code == 200:
                    print('✅ ¡URL conectada AUTOMÁTICAMENTE en Render!:', r_reg.json())
                    conectado = True
                    break
                print('Reintentando registro en Render:', r_reg.status_code, r_reg.text)
            except Exception as e:
                print('Reintentando registro en Render:', e)
            time.sleep(3)
    if not conectado:
        print(f'⚠️ Copia esta URL: {url_tunel} y pégala en {render_url}/estado')


In [ ]:
import numpy as np, cv2, requests
assert url_tunel, 'No se obtuvo URL de Cloudflare'
print('GET /health:', requests.get(url_tunel + '/health', timeout=10).json()['status'])
img_dummy = np.zeros((320, 480, 3), dtype=np.uint8)
ok, buf = cv2.imencode('.jpg', img_dummy)
headers = {'X-API-Key': os.environ['API_KEY']} if os.environ.get('API_KEY') else {}
r = requests.post(url_tunel + '/infer', files={'imagen': ('test.jpg', buf.tobytes(), 'image/jpeg')}, data={'modo': 'completo'}, headers=headers, timeout=30)
print('POST /infer status:', r.status_code, 'latencia_ms:', r.json().get('latencia_ms'))


In [ ]:
# Celda keep-alive: mantenla ejecutando durante la grabacion o demo
import time, requests
while True:
    try:
        requests.get('http://127.0.0.1:8000/health', timeout=5)
    except Exception:
        pass
    time.sleep(45)
